# Data preparation

In [26]:
import pandas as pd
import os

# Impute your path
work_dir =  os.getcwd()
directory = os.path.join(work_dir, '/content/drive/MyDrive/bioinf/glioblastoma/')
dfs = {}

# Download all files relevant to *_combined.tsv
for filename in os.listdir(directory):
    if filename.endswith('_final.tsv'):
        sample_name = filename.split('_final.tsv')[0]
        file_path = os.path.join(directory, filename)
        dfs[sample_name] = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1')

# Check DataFrames
for sample, df in dfs.items():
    print(f"DataFrame for {sample}:")
    print(df.head())

/tmp/ipython-input-26-644416338.py:14: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[sample_name] = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1')
/tmp/ipython-input-26-644416338.py:14: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[sample_name] = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1')
/tmp/ipython-input-26-644416338.py:14: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[sample_name] = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1')


DataFrame for LOC262:
  CHROM     POS ID REF ALT  QUAL                     FILTER LOC262_GT  \
0     1   63268  .   T   C    25                       PASS       0/1   
1     1  129285  .   G   A   105                       PASS       0/1   
2     1  183358  .   G   C    17                       PASS       0/1   
3     1  183531  .   G   A    24                       PASS       0/1   
4     1  183598  .   C   G    14  LowGQX;NoPassedVariantGTs       0/1   

  LOC262_DP Allele  ...                                         gnomADv4  \
0         8    C,C  ...                      1:63268-63268,1:63268-63268   
1        58    A,A  ...                  1:129285-129285,1:129285-129285   
2        21  C,C,C  ...  1:183358-183358,1:183358-183358,1:183358-183358   
3        32  A,A,A  ...  1:183531-183531,1:183531-183531,1:183531-183531   
4        53  G,G,G  ...  1:183598-183598,1:183598-183598,1:183598-183598   

                        gnomADv4_AF                gnomADv4_FAF  \
0              

The column CADD_phred needs to be converted to a numerical form so that it can filter

In [27]:
import numpy as np
import re

def extract_numbers(row):
    numbers = re.findall(r'\d+\.\d+', row)  #Find all numbers in row
    if numbers:
        return float(numbers[0])
    else:
        return np.nan

def fix_CADD_phred(df):
    df['CADD_phred'] = df['CADD_phred'].astype(str).apply(lambda x: extract_numbers(x))
    return df

for sample, df in dfs.items():
    dfs[sample] = fix_CADD_phred(df)

# Check the results
for sample, df in dfs.items():
    print(f"Processed CADD_phred for {sample}:")
    print(df['CADD_phred'].head())

Processed CADD_phred for LOC262:
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: CADD_phred, dtype: float64
Processed CADD_phred for PPL062:
0       NaN
1     2.209
2    14.750
3       NaN
4     8.789
Name: CADD_phred, dtype: float64
Processed CADD_phred for YAJ783:
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: CADD_phred, dtype: float64


[link text](https://)The column gnomADv_AF needs to be converted to a numerical form so that it can filter



In [29]:
def extract_numbers_to_null(row):
    numbers = re.findall(r'\d+\.?\d*(?:e-?\d+)?', row)  # Найти все числа в строке
    if numbers:
        return float(numbers[0])
    else:
        return 0

def fix_gnomADv4_AF(df):
    df['gnomADv4_AF'] = df['gnomADv4_AF'].astype(str).apply(lambda x: extract_numbers_to_null(x))
    return df

for sample, df in dfs.items():
    dfs[sample] = fix_gnomADv4_AF(df)

# Check the results
for sample, df in dfs.items():
    print(f"Processed gnomADv4_AF for {sample}:")
    print(df['gnomADv4_AF'].head())

Processed gnomADv4_AF for LOC262:
0    0.342735
1    0.644069
2    0.096632
3    0.009486
4    0.105530
Name: gnomADv4_AF, dtype: float64
Processed gnomADv4_AF for PPL062:
0    0.866202
1    0.951381
2    0.081296
3    0.671571
4    0.001451
Name: gnomADv4_AF, dtype: float64
Processed gnomADv4_AF for YAJ783:
0    0.532228
1    0.644069
2    0.000021
3    0.338472
4    0.177151
Name: gnomADv4_AF, dtype: float64


The letter values ​​corresponding to the tools for predicting pathogenic variants also need to be formatted.

In [30]:
def check_for_letters(value):
    if re.search(r'[a-zA-Z]', str(value)):
        return value
    else:
        return np.nan

columns = ['CLIN_SIG', 'DEOGEN2_pred', 'FATHMM_pred', 'LRT_pred', 'MetaSVM_pred', 'MutationTaster_pred', 'PROVEAN_pred', 'Polyphen2_HVAR_pred', 'PrimateAI_pred', 'SIFT_pred', 'CLINVAR_CLNSIG']


def fix_columns(df):
    df[columns] = df[columns].applymap(check_for_letters)
    return df

for sample, df in dfs.items():
    dfs[sample] = fix_columns(df)

# Check the results using the column FATHMM_pred as example
for sample, df in dfs.items():
    print(f"Processed FATHMM_pred for {sample}:")
    print(df['FATHMM_pred'].head())

/tmp/ipython-input-30-1008231097.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[columns] = df[columns].applymap(check_for_letters)
/tmp/ipython-input-30-1008231097.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[columns] = df[columns].applymap(check_for_letters)
/tmp/ipython-input-30-1008231097.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[columns] = df[columns].applymap(check_for_letters)


Processed FATHMM_pred for LOC262:
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: FATHMM_pred, dtype: object
Processed FATHMM_pred for PPL062:
0    NaN
1    .&T
2    .&T
3    NaN
4    .&T
Name: FATHMM_pred, dtype: object
Processed FATHMM_pred for YAJ783:
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: FATHMM_pred, dtype: object


# Filtration

In [31]:
def extract_numerator(val):
    if isinstance(val, str):
        match = re.search(r'(\d+)/(\d+)', val)
        if match:
            return float(match.group(1))
        try:
            return float(val)
        except:
            return np.nan
    else:
        return val


def gnomADv4_01_filtr(df):
    df = df.replace('.', np.nan)
    df.iloc[:, 8] = df.iloc[:, 8].replace(to_replace=r'^[\.,]+$', value=np.nan, regex=True)
    df.iloc[:, 8] = df.iloc[:, 8].apply(extract_numerator)
    df.iloc[:, 8] = df.iloc[:, 8].astype(float) #8= _DP, 7 = _GT
    df = df[(df.iloc[:, 7] != np.nan) & (df.iloc[:, 7] != '0/0') & (df['FILTER'] == 'PASS') & (df.iloc[:, 8] > 50) & (df['gnomADv4_AF'] <= 0.01)]
    return df

for sample, df in dfs.items():
    dfs[sample] = gnomADv4_01_filtr(df)

# Check the results
for sample, df in dfs.items():
    print(f"Filtered {sample} to a dataframe with the number of rows equal to:")
    print(len(df))

/tmp/ipython-input-31-14622147.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('.', np.nan)
/tmp/ipython-input-31-14622147.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('.', np.nan)
/tmp/ipython-input-31-14622147.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tru

Filtered LOC262 to a dataframe with the number of rows equal to:
2231
Filtered PPL062 to a dataframe with the number of rows equal to:
2203
Filtered YAJ783 to a dataframe with the number of rows equal to:
1837


Further filtering based on the pathogenicity criterion obtained from different tools - a variant was included in the sample if more than half of the tools had a pathogenicity prediction

In [32]:
thresholds = {
    'CADD_phred': 20
}


columns = ['CADD_phred', 'CLIN_SIG', 'CLINVAR_CLNSIG', 'DEOGEN2_pred', 'FATHMM_pred',
           'LRT_pred', 'MetaSVM_pred', 'MutationTaster_pred', 'PROVEAN_pred',
           'Polyphen2_HVAR_pred', 'PrimateAI_pred', 'SIFT_pred']


def is_pathogenic(row):
    pathogenic_count = 0
    count = 0

    if not pd.isna(row['CADD_phred']):
        if row['CADD_phred'] >= thresholds['CADD_phred']:
            pathogenic_count += 1
        count += 1

    if not pd.isna(row['CLIN_SIG']):
        if 'pathogenic' in row['CLIN_SIG'] or 'likely_pathogenic' in row['CLIN_SIG']:
            pathogenic_count += 1
        count += 1

    if not pd.isna(row['CLINVAR_CLNSIG']):
        if 'Pathogenic' in row['CLINVAR_CLNSIG'] or 'Likely_pathogenic' in row['CLINVAR_CLNSIG']:
            pathogenic_count += 1
        count += 1

    for col in columns[3:]:
        if not pd.isna(row[col]):
            if 'D' in str(row[col]):
                pathogenic_count += 1
            count += 1

    return pathogenic_count > (count / 2)


for sample, df in dfs.items():
    df['pathogenic'] = df.apply(is_pathogenic, axis=1)
    dfs[sample] = df

# Check the results
for sample, df in dfs.items():
    print(f"Filtered DataFrame for {sample} has {len(df)} rows.")
    print(df['pathogenic'].head())

Filtered DataFrame for LOC262 has 2231 rows.
159    False
273    False
281    False
301    False
302    False
Name: pathogenic, dtype: bool
Filtered DataFrame for PPL062 has 2203 rows.
149    False
186    False
242    False
325    False
814    False
Name: pathogenic, dtype: bool
Filtered DataFrame for YAJ783 has 1837 rows.
224    False
242    False
244    False
370    False
566    False
Name: pathogenic, dtype: bool


In [33]:
for sample, df in dfs.items():
    dfs[sample] = df[df['pathogenic'] == True]

# Check the results
for sample, df in dfs.items():
    print(f"Filtered DataFrame for {sample} has {len(df)} rows.")
    print(df['pathogenic'].head())

Filtered DataFrame for LOC262 has 81 rows.
6361     True
13681    True
18116    True
22491    True
24776    True
Name: pathogenic, dtype: bool
Filtered DataFrame for PPL062 has 80 rows.
1468    True
2216    True
5870    True
6166    True
7991    True
Name: pathogenic, dtype: bool
Filtered DataFrame for YAJ783 has 77 rows.
4397     True
5882     True
7410     True
8161     True
13258    True
Name: pathogenic, dtype: bool


In [34]:
for sample, df in dfs.items():
    df.to_csv(f'{sample}_pathogenic.csv', index=False)